MCP SCENARIO: “Smart IT Helpdesk Assistant”
🧩 Scenario Background

You are working in a company called ABC Corp.

Employees face issues like:

VPN not working
Printer not responding
Software errors

👉 Instead of calling IT support, employees use an AI Helpdesk Bot.

🤖 What this Bot Should Do

When a user types a problem:

Understand the issue
Decide if a ticket is needed
Identify:
Category (Network / Hardware / General)
Priority (High / Medium)
Create a ticket
Show confirmation
🧠 How MCP Fits Here
Component	Role in Scenario
Agent	Helpdesk Bot
MCP Layer	Decision + Tool calling
Tool	Ticket Creation System
User	Employee


In [1]:

# ============================================
# STEP 0: DATABASE (Simulated storage)
# ============================================

tickets_db = []  # This stores all tickets


# ============================================
# STEP 1: TOOL (MCP TOOL)
# ============================================

def create_ticket(issue, priority, category):
    """
    This function simulates a TOOL in MCP
    In real world → API / Database / ServiceNow
    """

    ticket_id = f"INC{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category
    }

    tickets_db.append(ticket)

    return ticket


# ============================================
# STEP 2: AGENT REASONING (LLM SIMULATION)
# ============================================

def analyze_input(user_input):
    """
    Simulates how an LLM understands user input
    Extracts:
    - category
    - priority
    """

    text = user_input.lower()

    # 🔹 Category Detection
    if "vpn" in text:
        category = "network"
    elif "printer" in text:
        category = "hardware"
    elif "email" in text:
        category = "software"
    else:
        category = "general"

    # 🔹 Priority Detection
    if "urgent" in text or "immediately" in text or 'fast' in text:
        priority = "high"
    elif "slow" in text:
        priority = "low"
    else:
        priority = "medium"

    return category, priority


# ============================================
# STEP 3: DECISION ENGINE (MCP CORE)
# ============================================

def should_call_tool(user_input):
    """
    Decides whether to call a tool or not
    This is MCP decision layer
    """

    keywords = ["issue", "problem", "ticket", "not working"]

    return any(word in user_input.lower() for word in keywords)


# ============================================
# STEP 4: MCP ORCHESTRATOR
# ============================================

def mcp_agent(user_input):
    """
    This is the MAIN MCP FLOW
    It connects:
    Agent → Decision → Tool → Response
    """

    print("\n🧠 Agent received input:", user_input)

    # STEP 4.1: Decision
    if should_call_tool(user_input):

        print("➡️ Decision: Tool call required")

        # STEP 4.2: Analyze input
        category, priority = analyze_input(user_input)

        print(f"📊 Extracted → Category: {category}, Priority: {priority}")

        # STEP 4.3: Prepare payload (MCP format)
        payload = {
            "issue": user_input,
            "priority": priority,
            "category": category
        }

        print("📦 MCP Payload:", payload)

        # STEP 4.4: Call tool
        result = create_ticket(**payload)

        print("⚙️ Tool executed successfully")

        # STEP 4.5: Final response
        return f"""
        ✅ Ticket Created Successfully!

        Ticket ID: {result['ticket_id']}
        Issue: {result['issue']}
        Category: {result['category']}
        Priority: {result['priority']}
        """

    else:
        print("➡️ Decision: No tool needed (AI response)")

        return "🤖 AI Response: Please describe your issue clearly."


# ============================================
# STEP 5: RUN INTERACTIVE LOOP
# ============================================

print("🚀 MCP Demo Started (Type 'exit' to stop)\n")

while True:

    user_input = input("Enter your query: ")

    if user_input.lower() == "exit":
        print("👋 Exiting MCP demo...")
        break

    response = mcp_agent(user_input)
    print(response)


🚀 MCP Demo Started (Type 'exit' to stop)


🧠 Agent received input: VpN not working do it fast
➡️ Decision: Tool call required
📊 Extracted → Category: network, Priority: medium
📦 MCP Payload: {'issue': 'VpN not working do it fast', 'priority': 'medium', 'category': 'network'}
⚙️ Tool executed successfully

        ✅ Ticket Created Successfully!

        Ticket ID: INC1000
        Issue: VpN not working do it fast
        Category: network
        Priority: medium
        

🧠 Agent received input: printer not working fix it fast 
➡️ Decision: Tool call required
📊 Extracted → Category: hardware, Priority: medium
📦 MCP Payload: {'issue': 'printer not working fix it fast ', 'priority': 'medium', 'category': 'hardware'}
⚙️ Tool executed successfully

        ✅ Ticket Created Successfully!

        Ticket ID: INC1001
        Issue: printer not working fix it fast 
        Category: hardware
        Priority: medium
        

🧠 Agent received input: wifi not working fix its urgent

➡️ Decision

In [4]:
import os
from groq import Groq
from dotenv import load_dotenv

load_dotenv()

# Load API key securely
api_key = os.getenv("GROQ_API_KEY")

client = Groq(api_key=api_key)

# ============================================
# STEP 0: DATABASE
# ============================================

tickets_db = []

# ============================================
# STEP 1: TOOL
# ============================================

def create_ticket(issue, priority, category):
    ticket_id = f"INC{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category
    }

    tickets_db.append(ticket)
    return ticket


# ============================================
# STEP 2: LLM ANALYSIS (REPLACES RULES)
# ============================================

def analyze_with_llm(user_input):
    """
    LLM decides:
    - should_create_ticket
    - category
    - priority
    """

    prompt = f"""
You are an IT helpdesk assistant.

Analyze the user issue and respond in JSON format:

{{
  "create_ticket": true/false,
  "category": "network/hardware/software/general",
  "priority": "high/medium/low"
}}

User Input: "{user_input}"
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",  # fast + powerful
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    output = response.choices[0].message.content

    try:
        import json
        parsed = json.loads(output)
    except:
        parsed = {
            "create_ticket": True,
            "category": "general",
            "priority": "medium"
        }

    return parsed


# ============================================
# STEP 3: MCP AGENT
# ============================================

def mcp_agent(user_input):

    print("\n🧠 Agent received:", user_input)

    # LLM Decision
    decision = analyze_with_llm(user_input)

    print("🤖 LLM Decision:", decision)

    if decision["create_ticket"]:

        payload = {
            "issue": user_input,
            "priority": decision["priority"],
            "category": decision["category"]
        }

        print("📦 MCP Payload:", payload)

        result = create_ticket(**payload)

        return f"""
✅ Ticket Created Successfully!

Ticket ID: {result['ticket_id']}
Issue: {result['issue']}
Category: {result['category']}
Priority: {result['priority']}
"""

    else:
        return "🤖 AI Response: No ticket required. Try basic troubleshooting."


# ============================================
# STEP 4: RUN LOOP
# ============================================

print("🚀 LLM MCP Helpdesk Started (type 'exit')\n")

while True:

    user_input = input("Enter issue: ")

    if user_input.lower() == "exit":
        print("👋 Exiting...")
        break

    response = mcp_agent(user_input)
    print(response)

🚀 LLM MCP Helpdesk Started (type 'exit')


🧠 Agent received: my camera not working 
🤖 LLM Decision: {'create_ticket': True, 'category': 'hardware', 'priority': 'medium'}
📦 MCP Payload: {'issue': 'my camera not working ', 'priority': 'medium', 'category': 'hardware'}

✅ Ticket Created Successfully!

Ticket ID: INC1000
Issue: my camera not working 
Category: hardware
Priority: medium


🧠 Agent received: my app is not working
🤖 LLM Decision: {'create_ticket': True, 'category': 'general', 'priority': 'medium'}
📦 MCP Payload: {'issue': 'my app is not working', 'priority': 'medium', 'category': 'general'}

✅ Ticket Created Successfully!

Ticket ID: INC1001
Issue: my app is not working
Category: general
Priority: medium


🧠 Agent received: vpn is not working need it to be fixed immeditely
🤖 LLM Decision: {'create_ticket': True, 'category': 'general', 'priority': 'medium'}
📦 MCP Payload: {'issue': 'vpn is not working need it to be fixed immeditely', 'priority': 'medium', 'category': 'general'}

MCP SCENARIO: “Smart HR Onboarding Assistant”
🧩 Scenario Background
You are working in a company called XYZ Corp.
New employees often face challenges during onboarding, such as:
- Trouble accessing payroll portal
- Confusion about leave policies
- Difficulty setting up email accounts
- Questions about training schedules
👉 Instead of emailing HR or waiting for responses, employees use an AI Onboarding Bot.

🤖 What this Bot Should Do
When a new hire types a question/problem:
- Understand the query (e.g., “I can’t log into payroll”)
- Decide if escalation to HR is needed
- Identify:
- Category (Payroll / Policy / IT Setup / Training)
- Priority (High / Medium)
- Create a support ticket if required
- Provide instant guidance (FAQs, step-by-step instructions)
- Show confirmation and next steps

🧠 How MCP Fits Here
|  |  |
|  |  |
|  |  |
|  |  |
|  |  |



This way, the MCP framework is reused in a Human Resources context, where the AI assistant streamlines onboarding, reduces HR workload, and ensures employees feel supported from day one.
Would you like me to design another variation in a customer service setting (like retail or banking), so you can compare how MCP adapts across industries?

In [5]:
# ============================================
# STEP 0: DATABASE
# ============================================

tickets_db = []


# ============================================
# STEP 1: TOOL
# ============================================

def create_ticket(issue, priority, category):
    ticket_id = f"HR{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category
    }

    tickets_db.append(ticket)
    return ticket


# ============================================
# STEP 2: ANALYZER (RULE-BASED)
# ============================================

def analyze_input(user_input):

    text = user_input.lower()

    # 🔹 Category Detection
    if "payroll" in text or "salary" in text:
        category = "Payroll"
    elif "leave" in text or "policy" in text:
        category = "Policy"
    elif "email" in text or "login" in text or "account" in text:
        category = "IT Setup"
    elif "training" in text:
        category = "Training"
    else:
        category = "General"

    # 🔹 Priority Detection
    if "urgent" in text or "immediately" in text:
        priority = "High"
    elif "not working" in text or "can't" in text:
        priority = "Medium"
    else:
        priority = "Low"

    return category, priority


# ============================================
# STEP 3: DECISION ENGINE
# ============================================

def should_create_ticket(user_input):

    keywords = ["not working", "can't", "issue", "problem", "error"]

    return any(word in user_input.lower() for word in keywords)


# ============================================
# STEP 4: FAQ / GUIDANCE ENGINE
# ============================================

def provide_guidance(category):

    if category == "Payroll":
        return "👉 Please log into payroll portal using your employee ID. Reset password if needed."
    elif category == "Policy":
        return "👉 Leave policy: 20 paid leaves/year. Check HR portal for details."
    elif category == "IT Setup":
        return "👉 Use your company email credentials. Contact IT if login fails."
    elif category == "Training":
        return "👉 Training schedule is available on onboarding dashboard."
    else:
        return "👉 Please check onboarding portal or contact HR."


# ============================================
# STEP 5: MCP AGENT
# ============================================

def mcp_agent(user_input):

    print("\n🧠 HR Bot received:", user_input)

    category, priority = analyze_input(user_input)

    print(f"📊 Category: {category}, Priority: {priority}")

    guidance = provide_guidance(category)

    if should_create_ticket(user_input):

        print("➡️ Decision: Creating ticket")

        result = create_ticket(user_input, priority, category)

        return f"""
✅ HR Ticket Created!

Ticket ID: {result['ticket_id']}
Category: {result['category']}
Priority: {result['priority']}

💡 Guidance:
{guidance}

📩 HR team will contact you shortly.
"""

    else:
        print("➡️ Decision: No ticket needed")

        return f"""
🤖 Instant Help:

{guidance}
"""


# ============================================
# STEP 6: RUN LOOP
# ============================================

print("🚀 HR Onboarding Assistant Started (type 'exit')\n")

while True:
    user_input = input("Ask HR Bot: ")

    if user_input.lower() == "exit":
        print("👋 Exiting...")
        break

    print(mcp_agent(user_input))

🚀 HR Onboarding Assistant Started (type 'exit')


🧠 HR Bot received: i want leave in urgent
📊 Category: Policy, Priority: High
➡️ Decision: No ticket needed

🤖 Instant Help:

👉 Leave policy: 20 paid leaves/year. Check HR portal for details.


🧠 HR Bot received: email not working need urgent
📊 Category: IT Setup, Priority: High
➡️ Decision: Creating ticket

✅ HR Ticket Created!

Ticket ID: HR1000
Category: IT Setup
Priority: High

💡 Guidance:
👉 Use your company email credentials. Contact IT if login fails.

📩 HR team will contact you shortly.

👋 Exiting...


In [ ]:
import os
import json
from groq import Groq
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")
client = Groq(api_key=api_key)


# ============================================
# STEP 0: DATABASE
# ============================================

tickets_db = []


# ============================================
# STEP 1: TOOL
# ============================================

def create_ticket(issue, priority, category):

    ticket_id = f"HR{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category
    }

    tickets_db.append(ticket)
    return ticket


# ============================================
# STEP 2: LLM ANALYSIS
# ============================================

def analyze_with_llm(user_input):

    prompt = f"""
You are an HR onboarding assistant.

Analyze the employee query and respond in JSON format:

{{
  "create_ticket": true/false,
  "category": "Payroll/Policy/IT Setup/Training/General",
  "priority": "High/Medium/Low",
  "guidance": "short helpful response for user"
}}

Rules:
- Create ticket ONLY if issue/problem exists
- High priority only for urgent problems
- Always give helpful guidance

User Input: "{user_input}"
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    output = response.choices[0].message.content

    try:
        parsed = json.loads(output)
    except:
        parsed = {
            "create_ticket": True,
            "category": "General",
            "priority": "Medium",
            "guidance": "Please contact HR for assistance."
        }

    return parsed


# ============================================
# STEP 3: MCP AGENT
# ============================================

def mcp_agent(user_input):

    print("\n🧠 HR Bot received:", user_input)

    decision = analyze_with_llm(user_input)

    print("🤖 LLM Decision:", decision)

    if decision["create_ticket"]:

        result = create_ticket(
            user_input,
            decision["priority"],
            decision["category"]
        )

        return f"""
✅ HR Ticket Created!

Ticket ID: {result['ticket_id']}
Category: {result['category']}
Priority: {result['priority']}

💡 Guidance:
{decision['guidance']}

📩 HR will reach out soon.
"""

    else:
        return f"""
🤖 Instant Help:

{decision['guidance']}
"""


# ============================================
# STEP 4: RUN LOOP
# ============================================

print("🚀 AI HR Onboarding Assistant Started (type 'exit')\n")

while True:

    user_input = input("Ask HR Bot: ")

    if user_input.lower() == "exit":
        print("👋 Exiting...")
        break

    print(mcp_agent(user_input))

MCP SCENARIO: “Smart Banking Support Assistant”
🧩 Scenario Background
You are working in a company called FinTrust Bank.
Customers often face issues such as:
- Credit card not working
- Trouble with online banking login
- Queries about loan status
- Transaction disputes
👉 Instead of calling customer care, customers use an AI Banking Support Bot.

🤖 What this Bot Should Do
When a customer types a problem:
- Understand the issue (e.g., “My card was declined”)
- Decide if escalation to a human agent is needed
- Identify:
- Category (Card Services / Online Banking / Loans / Transactions)
- Priority (High / Medium)
- Create a support ticket if required
- Provide instant guidance (FAQs, troubleshooting steps, policy info)
- Show confirmation and next steps


This way, MCP is applied in a financial services context, where the AI assistant reduces call center load, provides quick resolutions, and ensures customers feel supported with secure, reliable guidance.
Would you like me to craft one more in a healthcare setting (like hospital patient support), so you can see how MCP adapts to critical service environments?

In [1]:
# ============================================
# STEP 0: DATABASE
# ============================================

tickets_db = []


# ============================================
# STEP 1: TOOL
# ============================================

def create_ticket(issue, priority, category):
    ticket_id = f"BNK{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category
    }

    tickets_db.append(ticket)
    return ticket


# ============================================
# STEP 2: ANALYZER (RULE-BASED)
# ============================================

def analyze_input(user_input):

    text = user_input.lower()

    # 🔹 Category Detection
    if "card" in text or "credit" in text or "debit" in text:
        category = "Card Services"
    elif "login" in text or "online banking" in text:
        category = "Online Banking"
    elif "loan" in text:
        category = "Loans"
    elif "transaction" in text or "payment" in text or "failed" in text:
        category = "Transactions"
    else:
        category = "General"

    # 🔹 Priority Detection
    if "urgent" in text or "immediately" in text or "declined" in text:
        priority = "High"
    else:
        priority = "Medium"

    return category, priority


# ============================================
# STEP 3: DECISION ENGINE
# ============================================

def should_create_ticket(user_input):

    keywords = ["not working", "declined", "failed", "issue", "problem", "error"]

    return any(word in user_input.lower() for word in keywords)


# ============================================
# STEP 4: GUIDANCE ENGINE
# ============================================

def provide_guidance(category):

    if category == "Card Services":
        return "👉 Check if your card is active and has sufficient balance. Try again or contact bank if issue persists."
    elif category == "Online Banking":
        return "👉 Reset your password using 'Forgot Password' or ensure correct login credentials."
    elif category == "Loans":
        return "👉 Check your loan status in the loan section of the banking app."
    elif category == "Transactions":
        return "👉 Wait for 24 hours for pending transactions. Contact support if amount is debited."
    else:
        return "👉 Please visit the help section or contact customer support."


# ============================================
# STEP 5: MCP AGENT
# ============================================

def mcp_agent(user_input):

    print("\n🧠 Banking Bot received:", user_input)

    category, priority = analyze_input(user_input)

    print(f"📊 Category: {category}, Priority: {priority}")

    guidance = provide_guidance(category)

    if should_create_ticket(user_input):

        print("➡️ Decision: Creating ticket")

        result = create_ticket(user_input, priority, category)

        return f"""
✅ Support Ticket Created!

Ticket ID: {result['ticket_id']}
Category: {result['category']}
Priority: {result['priority']}

💡 Guidance:
{guidance}

📩 Our banking team will contact you shortly.
"""

    else:
        print("➡️ Decision: No ticket needed")

        return f"""
🤖 Instant Help:

{guidance}
"""


# ============================================
# STEP 6: RUN LOOP
# ============================================

print("🚀 Banking Support Assistant Started (type 'exit')\n")

while True:
    user_input = input("Ask Banking Bot: ")

    if user_input.lower() == "exit":
        print("👋 Exiting...")
        break

    print(mcp_agent(user_input))

🚀 Banking Support Assistant Started (type 'exit')


🧠 Banking Bot received: i want to take a loan urgently
📊 Category: Loans, Priority: High
➡️ Decision: No ticket needed

🤖 Instant Help:

👉 Check your loan status in the loan section of the banking app.


🧠 Banking Bot received: i cannot access the bank account
📊 Category: General, Priority: Medium
➡️ Decision: No ticket needed

🤖 Instant Help:

👉 Please visit the help section or contact customer support.


🧠 Banking Bot received: login not working
📊 Category: Online Banking, Priority: Medium
➡️ Decision: Creating ticket

✅ Support Ticket Created!

Ticket ID: BNK1000
Category: Online Banking
Priority: Medium

💡 Guidance:
👉 Reset your password using 'Forgot Password' or ensure correct login credentials.

📩 Our banking team will contact you shortly.

👋 Exiting...


In [ ]:
import os
import json
from groq import Groq
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")
client = Groq(api_key=api_key)


# ============================================
# STEP 0: DATABASE
# ============================================

tickets_db = []


# ============================================
# STEP 1: TOOL
# ============================================

def create_ticket(issue, priority, category):

    ticket_id = f"BNK{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category
    }

    tickets_db.append(ticket)
    return ticket


# ============================================
# STEP 2: LLM ANALYSIS
# ============================================

def analyze_with_llm(user_input):

    prompt = f"""
You are a banking support assistant.

Analyze the customer query and respond in JSON format:

{{
  "create_ticket": true/false,
  "category": "Card Services/Online Banking/Loans/Transactions/General",
  "priority": "High/Medium",
  "guidance": "short helpful response for user"
}}

Rules:
- Create ticket ONLY if real issue exists
- High priority for declined transactions or urgent issues
- Always provide safe and helpful banking guidance

User Input: "{user_input}"
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    output = response.choices[0].message.content

    try:
        parsed = json.loads(output)
    except:
        parsed = {
            "create_ticket": True,
            "category": "General",
            "priority": "Medium",
            "guidance": "Please contact customer support for further assistance."
        }

    return parsed


# ============================================
# STEP 3: MCP AGENT
# ============================================

def mcp_agent(user_input):

    print("\n🧠 Banking Bot received:", user_input)

    decision = analyze_with_llm(user_input)

    print("🤖 LLM Decision:", decision)

    if decision["create_ticket"]:

        result = create_ticket(
            user_input,
            decision["priority"],
            decision["category"]
        )

        return f"""
✅ Support Ticket Created!

Ticket ID: {result['ticket_id']}
Category: {result['category']}
Priority: {result['priority']}

💡 Guidance:
{decision['guidance']}

📩 Our banking team will contact you shortly.
"""

    else:
        return f"""
🤖 Instant Help:

{decision['guidance']}
"""


# ============================================
# STEP 4: RUN LOOP
# ============================================

print("🚀 AI Banking Support Assistant Started (type 'exit')\n")

while True:

    user_input = input("Ask Banking Bot: ")

    if user_input.lower() == "exit":
        print("👋 Exiting...")
        break

    print(mcp_agent(user_input))

Create a  Weather Tool MCP Server that any AI agent can use with sample use case

In [7]:
# ================== INSTALL (run once) ==================

# ================== IMPORTS ==================
from groq import Groq
import requests
import json
import os
from dotenv import load_dotenv
load_dotenv()

# ================== CONFIG ==================
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
MCP_SERVER = "http://127.0.0.1:8000"  # Make sure your MCP server is running!

client = Groq(api_key=GROQ_API_KEY)

# ================== TOOLS ==================
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get current weather for a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {"type": "string"}
                },
                "required": ["location"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_forecast",
            "description": "Get 3-day weather forecast for a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {"type": "string"}
                },
                "required": ["location"]
            }
        }
    }
]

# ================== SMART ADVICE ==================
def weather_advice(data):
    temp = data.get("temperature_c", 0)

    if temp > 35:
        return "🔥 It's very hot. Avoid going out!"
    elif temp > 30:
        return "🌤 Warm weather. Stay hydrated!"
    elif temp > 20:
        return "😊 Pleasant weather!"
    else:
        return "🧥 Might be chilly."

# ================== AGENT ==================
def run_agent(user_query):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": user_query}],
        tools=tools,
        tool_choice="auto"
    )

    msg = response.choices[0].message

    # 🔧 Tool Call Handling
    if msg.tool_calls:
        tool_call = msg.tool_calls[0]
        function_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)

        api_response = requests.post(
            f"{MCP_SERVER}/tools/{function_name}",
            json=args
        ).json()

        # 🌤 Current Weather Output
        if function_name == "get_current_weather":
            advice = weather_advice(api_response)

            return f"""
📍 Location: {api_response.get('location')}
🌡 Temperature: {api_response.get('temperature_c')}°C
🤒 Feels Like: {api_response.get('feels_like_c')}°C
🌥 Condition: {api_response.get('condition')}
💧 Humidity: {api_response.get('humidity')}%

👉 Advice: {advice}
"""

        # 🌦 Forecast Output
        elif function_name == "get_forecast":
            forecast_text = "📊 3-Day Forecast:\n\n"

            for day in api_response.get("forecast", []):
                forecast_text += f"""
📅 {day['date']}
🌡 Avg: {day['avg_temp_c']}°C
🔺 Max: {day['max_temp_c']}°C
🔻 Min: {day['min_temp_c']}°C
🌥 {day['condition']}

"""
            return forecast_text

    return msg.content

# ================== TEST ==================
print(run_agent("What's the forecast of weather in greater noida?"))

📊 3-Day Forecast:


📅 2026-03-28
🌡 Avg: 27.7°C
🔺 Max: 33.2°C
🔻 Min: 22.2°C
🌥 Sunny


📅 2026-03-29
🌡 Avg: 30.1°C
🔺 Max: 36.2°C
🔻 Min: 23.4°C
🌥 Patchy rain nearby


📅 2026-03-30
🌡 Avg: 30.0°C
🔺 Max: 35.7°C
🔻 Min: 26.7°C
🌥 Patchy rain nearby


